# PPO를 직접 구현하며 이해하기 — Space Z / TankEnv
**목표:** Stable-Baselines3의 `PPO` 클래스를 가져오지 않고, 현재 프로젝트의 실제 `TankEnv`를 이용해 PPO 핵심 동작을 구현한다.

읽는 순서:
1. **가우시안 행동**: Actor가 '정답 행동'을 직접 출력하는 대신 평균·표준편차를 만들고 샘플링한다.
2. **Critic과 explained variance**: 상태 가치 예측이 Return의 변동을 얼마나 설명하는지 확인한다.
3. **rollout → GAE → clipped PPO loss → 역전파**를 실제 게임 환경에서 연결한다.
4. 학습 전후 지표를 읽고, 실제 환경에서 행동을 재실행한다.

> 교육용 축약 구현이다. SB3 전체 기능(벡터 병렬환경, 스케줄, gSDE, 관측 이미지 정책, 다양한 정규화·로깅, 완전한 체크포인트 재개 등)을 재현하지 않는다. `MlpPolicy` + 5차원 `Box` 행동 환경을 대상으로 한다.

## 1. 준비: 프로젝트와 현재 게임 환경

노트북은 `study/`에서 실행한다고 가정하지만, 실행 디렉터리가 달라도 프로젝트 루트를 위로 탐색한다. `train.py`와 **같은 게임·관측·행동·보상 설정**을 사용한다. 학습 속도를 위해 화면 렌더링은 끈다.

In [2]:
START

WindowsPath('C:/LANG_CHAIN_2026/final-project/2026-10-12_game/study')

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.distributions import Normal
from torch.nn import functional as F

START = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (START, *START.parents)
     if (folder / "src/space_z/rl/environment.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. 프로젝트 내부에서 노트북을 실행하세요.")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from space_z.config import GameConfig, RewardConfig, EnvConfig
from space_z.rl.actions.continuous_action_decoder import ContinuousActionDecoder
from space_z.rl.environment import TankEnv
from space_z.rl.observations.vector_observation import VectorObservation
from space_z.rl.rewards import RewardCalculator

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(1)

DEVICE = torch.device("cpu")
N_STEPS = 256
BATCH_SIZE = 64
N_EPOCHS = 10
N_ITERATIONS = 3  # 구조 학습용. 좋은 게임 성능을 기대할 만큼 긴 학습은 아니다.
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_RANGE = 0.2
VALUE_COEF = 0.5
ENTROPY_COEF = 0.0
MAX_GRAD_NORM = 0.5
LEARNING_RATE = 3e-4
print("Project:", PROJECT_ROOT)

Project: C:\LANG_CHAIN_2026\final-project\2026-10-12_game


In [ ]:
def make_environment() -> TankEnv:
    """train.py의 벡터 학습 환경과 같은 구성. PPO 내부 구현은 이 환경을 변경하지 않는다."""
    reward_config = RewardConfig(
        shape_damage=0.02,
        shape_destroy=0.04,
        enemy_damage=0.04,
        enemy_kill=2.0,
        damage_taken=-0.02, 
        death=-20.0,
        time_step=-0.0001,
        shot_cost=0.0001,
    )
    return TankEnv(
        GameConfig(),
        VectorObservation(),
        ContinuousActionDecoder(enable_stat_upgrades=True, enable_evolution=True),
        RewardCalculator(reward_config),
        env_config=EnvConfig(frame_skip=4, max_steps=3000),
    )

environment = make_environment()
initial_observation, initial_info = environment.reset(seed=SEED)

observation_dim = int(np.prod(environment.observation_space.shape))
action_dim = int(np.prod(environment.action_space.shape))

print("관측:", environment.observation_space.shape, "행동:", environment.action_space.shape)
print("한 환경 step의 최대 게임 프레임:", environment.environment_config.frame_skip)
print("실제 첫 관측:", initial_observation.shape)
assert observation_dim == 5139 and action_dim == 5
assert environment.observation_space.contains(initial_observation)

## 2. 가우시안(Normal)에서 행동을 뽑는다는 뜻

연속형 PPO의 Actor는 행동 5개를 직접 고정하는 대신, **각 차원에 대한 평균 `μ`와 표준편차 `σ`**를 정한다.

\[
a_i \sim \mathcal N(\mu_i(s), \sigma_i^2), \qquad
\log\pi(a\mid s)=\sum_{i=1}^{5}\log\mathcal N(a_i;\mu_i,\sigma_i^2)
\]

같은 상태에서도 다른 행동이 나올 수 있다. 표준편차가 작아지면 평균 근처의 행동이 더 자주 나온다.

**중요:** 현재 게임의 `Box(-1,1,(5,))` 제어값은 실제로 **이동(9구간), 회전(연속), 발사(0 초과), 스탯(구간), 전직(구간)**으로 해석된다. 모델에서 뽑은 원래 행동이 -1~1 밖이면 환경에 보내기 전 잘라준다. PPO의 로그 확률 계산에는 **원본 행동**을 사용한다.

In [ ]:
normal_example = Normal(loc=torch.tensor(-0.3), scale=torch.tensor(1.0))
example_actions = normal_example.sample((20_000,)).numpy()

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(example_actions, bins=60, density=True, alpha=0.8)
ax.axvline(-0.3, linestyle="--", label="mean = -0.3")
ax.axvline(-1, linestyle=":", label="action bounds")
ax.axvline(1, linestyle=":")
ax.set(title="Gaussian action samples (one action dimension)",
       xlabel="raw sampled action", ylabel="density")
ax.legend()
plt.show()

sample = normal_example.sample((8,))
print("샘플:", np.round(sample.numpy(), 2))
print("게임에 입력할 값:", np.round(sample.clamp(-1, 1).numpy(), 2))
print("범위 밖 비율:", float((sample.abs() > 1).float().mean()))

In [ ]:
example_decoder = environment.action_decoder
for sample_action in [
    np.array([-1.0, 0.0, -1.0, -1.0, -1.0], dtype=np.float32),
    np.array([ 0.2, 0.5,  0.7,  0.1, -0.3], dtype=np.float32),
]:
    print("행동:", sample_action, "→ 게임 제어:", example_decoder.decode(sample_action))

## 3. Actor-Critic 신경망 구성

현재 SB3 출력의 `5139 → 64 → 64` 두 갈래를 그대로 모방한다.

- **Actor:** `state → policy_net → action_mean(5)`, 별도 학습 파라미터 `log_std(5)`를 통해 Normal 분포 생성
- **Critic:** `state → value_net → V(s)(1)`
- `log_std`는 상태별 출력이 아니라 학습 가능한 공통 파라미터다. SB3의 기본 `DiagGaussianDistribution`과 비슷한 구조다.

**차원:** 입력은 5,139개지만 행동은 5차원이며, 5개 모두 연속 분포에서 먼저 샘플링한다.

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, observation_dim: int, action_dim: int):
        super().__init__()

        self.policy_net = nn.Sequential(
            nn.Linear(observation_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
        )
        self.value_features = nn.Sequential(
            nn.Linear(observation_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
        )
        self.action_net = nn.Linear(64, action_dim)
        self.value_net = nn.Linear(64, 1)
        self.log_std = nn.Parameter(torch.zeros(action_dim))

    def distribution_and_value(self, observations: torch.Tensor):
        """배치 상태를 받아 행동 분포와 상태 가치 추정치를 함께 반환."""
        action_mean = self.action_net(self.policy_net(observations))
        action_std = self.log_std.exp().expand_as(action_mean)
        distribution = Normal(action_mean, action_std)
        state_value = self.value_net(self.value_features(observations)).squeeze(-1)
        return distribution, state_value

policy = ActorCritic(observation_dim, action_dim).to(DEVICE)
optimizer = torch.optim.Adam(policy.parameters(), lr=LEARNING_RATE)

print(policy)
print("학습 파라미터 수:", sum(p.numel() for p in policy.parameters()))

### Actor의 행동이 게임에 들어가는 과정

`Normal.sample()`은 5개의 실수를 생성한다. **원본 행동**의 로그 확률을 보관하고, `np.clip`으로 `[-1,1]`을 지킨 **실행 행동**만 환경에 전달한다. 학습 때 잘라낸 값으로 로그 확률을 다시 계산하면 PPO 비율 계산이 어긋난다.

In [ ]:
def choose_action(policy: ActorCritic, observation: np.ndarray):
    """상태 → 원본 행동 / 게임 실행 행동 / 로그 확률 / 상태 가치."""
    state = torch.as_tensor(observation, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    with torch.no_grad():
        distribution, state_value = policy.distribution_and_value(state)
        raw_action_tensor = distribution.sample()
        log_probability = distribution.log_prob(raw_action_tensor).sum(dim=-1)

    raw_action = raw_action_tensor.squeeze(0).cpu().numpy().astype(np.float32)
    env_action = np.clip(
        raw_action, environment.action_space.low, environment.action_space.high
    ).astype(np.float32)

    return raw_action, env_action, float(log_probability.item()), float(state_value.item())

raw_action, env_action, log_probability, state_value = choose_action(policy, initial_observation)
print("원본 행동:", np.round(raw_action, 3))
print("실행 행동:", np.round(env_action, 3))
print("게임 해석:", environment.action_decoder.decode(env_action))
print("log π(a|s):", round(log_probability, 3), "| V(s):", round(state_value, 3))

## 4. explained_variance는 무엇일까?

Critic이 예측한 `V(s)`와 실제 학습 목표인 **Return**을 비교하는 지표다.

\[
\mathrm{EV} = 1 - \frac{\mathrm{Var}(\text{Return} - V(s))}{\mathrm{Var}(\text{Return})}
\]

- `1`: Return 변화를 완벽하게 설명
- `0`: 일정한 값으로 예측한 기준과 같은 수준
- `음수`: 그 기준보다 변동 패턴의 설명력이 낮음

`value_loss`는 **오차의 크기(MSE)**, EV는 **Return 변동을 얼마나 설명하는지**를 본다. Return 분산이 0이면 EV는 정의되지 않는다.

In [ ]:
def explained_variance(returns: np.ndarray, predictions: np.ndarray) -> float:
    returns = np.asarray(returns, dtype=np.float64)
    predictions = np.asarray(predictions, dtype=np.float64)
    return_variance = np.var(returns)
    if return_variance == 0:
        return float("nan")
    return float(1 - np.var(returns - predictions) / return_variance)

toy_returns = np.array([1., 2., 3., 4.])
for name, guess in {
    "완벽한 예측": toy_returns,
    "평균만 예측": np.full(4, toy_returns.mean()),
    "반대 방향 예측": -toy_returns,
}.items():
    print(f"{name:12} explained_variance = {explained_variance(toy_returns, guess):.3f}")

## 5. 256-step 경험 수집 (rollout)

PPO의 첫 단계는 게임에서 실제로 행동해 경험을 모으는 것이다. 한 데이터는 다음 형태다.

`(s_t, raw_action_t, logπ_old, V(s_t), reward_t, terminated_t, truncated_t, V(s_{t+1}))`

- 1개 환경 step은 기본적으로 게임 4프레임을 진행한다(조기 사망 시 짧음).
- 종료하면 `reset()`하여 다음 에피소드를 수집한다. 256-step은 **에피소드 길이와 별개**다.
- `terminated`면 다음 가치 0, `truncated`면 다음 관측에 대한 가치를 bootstrap한다.
- 이 구현은 한 환경만 사용한다. 완성형 SB3의 벡터화·모니터링 등은 생략한다.

In [ ]:
def collect_rollout(
    env: TankEnv,
    model: ActorCritic,
    current_observation: np.ndarray,
    running_reward: float,
    running_length: int,
):
    """정확히 N_STEPS개의 경험을 만들고, 이어서 실행할 상태도 반환."""
    keys = ("states", "raw_actions", "old_log_probs", "values",
            "rewards", "terminated", "done", "next_values")
    rollout = {key: [] for key in keys}
    completed_episodes = []
    clipped_action_elements = 0

    for _ in range(N_STEPS):
        state_tensor = torch.as_tensor(
            current_observation, dtype=torch.float32, device=DEVICE
        ).unsqueeze(0)
        with torch.no_grad():
            distribution, value = model.distribution_and_value(state_tensor)
            raw_action_tensor = distribution.sample()
            old_log_prob = distribution.log_prob(raw_action_tensor).sum(dim=-1)

        raw_action = raw_action_tensor.squeeze(0).cpu().numpy().astype(np.float32)
        env_action = np.clip(raw_action, env.action_space.low, env.action_space.high).astype(np.float32)
        clipped_action_elements += int(np.count_nonzero(raw_action != env_action))

        next_observation, reward, terminated, truncated, info = env.step(env_action)
        done = terminated or truncated
        with torch.no_grad():
            next_state_tensor = torch.as_tensor(
                next_observation, dtype=torch.float32, device=DEVICE
            ).unsqueeze(0)
            _, next_value_tensor = model.distribution_and_value(next_state_tensor)
        # 시간 제한 종료(truncated)는 bootstrap하되, 실제 사망(terminated)은 0
        next_value = 0.0 if terminated else float(next_value_tensor.item())

        rollout["states"].append(np.asarray(current_observation, dtype=np.float32))
        rollout["raw_actions"].append(raw_action)
        rollout["old_log_probs"].append(float(old_log_prob.item()))
        rollout["values"].append(float(value.item()))
        rollout["rewards"].append(float(reward))
        rollout["terminated"].append(bool(terminated))
        rollout["done"].append(bool(done))
        rollout["next_values"].append(next_value)

        running_reward += float(reward)
        running_length += 1
        current_observation = next_observation

        if done:
            completed_episodes.append({
                "reward": running_reward, "steps": running_length,
                "score": info["score"], "kills": info["kills"],
                "game_frames": info["frames"],
            })
            current_observation, _ = env.reset()
            running_reward, running_length = 0.0, 0

    rollout = {key: np.asarray(values) for key, values in rollout.items()}
    rollout["clipped_fraction"] = clipped_action_elements / (N_STEPS * action_dim)
    return rollout, current_observation, running_reward, running_length, completed_episodes

## 6. GAE — 보상과 Critic 예측으로 Advantage 계산

PPO가 Actor에 주는 신호는 **행동이 예상보다 좋았는가?** 라는 Advantage다.

\[
\delta_t=r_t+\gamma V(s_{t+1})-V(s_t),\quad
A_t=\delta_t+\gamma\lambda A_{t+1}
\]

두 번째 식은 동일 에피소드 내에서 뒤에서 앞으로 계산한다. `terminated`(사망)는 다음 가치가 0, `truncated`(시간 제한)는 다음 가치를 bootstrap하지만 에피소드 경계를 넘어 GAE를 이어 붙이지 않는다.

`return_target = A + V(s)`가 Critic의 학습 목표다.

In [ ]:
def calculate_gae(rollout: dict):
    advantages = np.zeros(N_STEPS, dtype=np.float32)
    next_advantage = 0.0

    for index in reversed(range(N_STEPS)):
        future_value = 0.0 if rollout["terminated"][index] else rollout["next_values"][index]
        delta = (
            rollout["rewards"][index]
            + GAMMA * future_value
            - rollout["values"][index]
        )
        continuation = 0.0 if rollout["done"][index] else 1.0
        next_advantage = delta + GAMMA * GAE_LAMBDA * continuation * next_advantage
        advantages[index] = next_advantage

    return_targets = advantages + rollout["values"]
    return advantages, return_targets

## 7. PPO의 핵심: 확률 비율을 클리핑하여 학습

rollout 때 보관한 구정책 확률과, 현재 업데이트된 Actor 확률을 비교한다.

\[
r_t=\exp(\log\pi_{new}(a_t|s_t)-\log\pi_{old}(a_t|s_t))
\]

\[
L_{actor}=-\mathbb{E}\left[\min(r_t A_t,\,\mathrm{clip}(r_t,0.8,1.2)A_t)\right]
\]

`clip`은 **행동값 [-1,1] 클리핑과 다른 개념**이다. 정책 확률 비율을 손실 계산에서 제한해 한번에 정책이 크게 바뀌는 이득을 억제한다.

최종 Loss = Actor Loss + 0.5 × Critic MSE − ent_coef × Entropy. 현재 `ent_coef=0.0`이며, Normal 분포의 `log_std`도 Actor Loss의 기울기로 업데이트된다.

In [ ]:
random_generator = np.random.default_rng(SEED)

def update_policy(model: ActorCritic, optimizer: torch.optim.Optimizer, rollout: dict):
    """한 rollout을 N_EPOCHS번 미니배치 학습. 최신 정책의 진단값도 반환."""
    advantages, return_targets = calculate_gae(rollout)

    states = torch.as_tensor(rollout["states"], dtype=torch.float32, device=DEVICE)
    raw_actions = torch.as_tensor(rollout["raw_actions"], dtype=torch.float32, device=DEVICE)
    old_log_probs = torch.as_tensor(rollout["old_log_probs"], dtype=torch.float32, device=DEVICE)
    return_tensors = torch.as_tensor(return_targets, dtype=torch.float32, device=DEVICE)
    advantage_tensors = torch.as_tensor(advantages, dtype=torch.float32, device=DEVICE)

    # Actor 학습의 수치 안정화. Critic의 return_target 자체를 정규화하지는 않는다.
    advantage_tensors = (
        advantage_tensors - advantage_tensors.mean()
    ) / (advantage_tensors.std(unbiased=False) + 1e-8)

    with torch.no_grad():
        _, before_values = model.distribution_and_value(states)
        before_ev = explained_variance(return_targets, before_values.cpu().numpy())

    all_policy_losses, all_value_losses = [], []
    for epoch in range(N_EPOCHS):
        shuffled = random_generator.permutation(N_STEPS)
        for start in range(0, N_STEPS, BATCH_SIZE):
            indices = shuffled[start:start + BATCH_SIZE]
            distribution, values = model.distribution_and_value(states[indices])
            new_log_probs = distribution.log_prob(raw_actions[indices]).sum(dim=-1)

            log_ratio = new_log_probs - old_log_probs[indices]
            ratio = torch.exp(log_ratio)
            unclipped = ratio * advantage_tensors[indices]
            clipped = torch.clamp(ratio, 1-CLIP_RANGE, 1+CLIP_RANGE) * advantage_tensors[indices]
            actor_loss = -torch.min(unclipped, clipped).mean()
            critic_loss = F.mse_loss(values, return_tensors[indices])
            entropy = distribution.entropy().sum(dim=-1).mean()

            total_loss = actor_loss + VALUE_COEF * critic_loss - ENTROPY_COEF * entropy
            optimizer.zero_grad(set_to_none=True)
            total_loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()

            all_policy_losses.append(float(actor_loss.item()))
            all_value_losses.append(float(critic_loss.item()))

    with torch.no_grad():
        distribution, after_values = model.distribution_and_value(states)
        log_ratio = distribution.log_prob(raw_actions).sum(-1) - old_log_probs
        ratio = torch.exp(log_ratio)
        approx_kl = ((ratio - 1) - log_ratio).mean().item()
        clip_fraction = (torch.abs(ratio-1) > CLIP_RANGE).float().mean().item()

    return {
        "actor_loss": float(np.mean(all_policy_losses)),
        "critic_loss": float(np.mean(all_value_losses)),
        "approx_kl": approx_kl,
        "clip_fraction": clip_fraction,
        "explained_variance_before": before_ev,
        "explained_variance_after": explained_variance(
            return_targets, after_values.cpu().numpy()
        ),
        "std_mean": float(model.log_std.exp().mean().item()),
        "mean_step_reward": float(rollout["rewards"].mean()),
        "action_clip_fraction": float(rollout["clipped_fraction"]),
    }

## 8. 실제 TankEnv에서 몇 차례 학습해보기

각 반복은 **256 × 4 = 최대 1,024 게임 프레임**을 진행한다. rollout이 끝날 때마다 `GAE → PPO 업데이트`를 수행한다. 이 짧은 예제는 동작을 관찰하는 목적이므로 평균 보상이나 EV 개선이 보장되지 않는다.

In [ ]:
class MiniPPO:
    """PPO 클래스가 담당하는 '경험 수집 → 정책 업데이트' 흐름을 축약한 실행기."""

    def __init__(self, environment: TankEnv, model: ActorCritic, optimizer):
        self.environment = environment
        self.model = model
        self.optimizer = optimizer

    def train_iteration(self, observation, running_reward, running_length):
        rollout, next_observation, next_reward, next_length, episodes = collect_rollout(
            self.environment, self.model, observation, running_reward, running_length
        )
        metrics = update_policy(self.model, self.optimizer, rollout)
        return next_observation, next_reward, next_length, episodes, metrics


trainer = MiniPPO(environment, policy, optimizer)

observation = initial_observation
running_episode_reward = 0.0
running_episode_length = 0
training_history = []

for iteration in range(1, N_ITERATIONS + 1):
    observation, running_episode_reward, running_episode_length, episodes, metrics = trainer.train_iteration(
        observation, running_episode_reward, running_episode_length
    )
    training_history.append(metrics)

    episode_message = (
        f"완료 에피소드 {len(episodes)}, 평균 에피소드 보상 "
        f"{np.mean([ep['reward'] for ep in episodes]):.2f}"
        if episodes else "완료 에피소드 없음(진행 중)"
    )
    print(
        f"[{iteration:2d}/{N_ITERATIONS}] "
        f"steps={iteration*N_STEPS:5d} | {episode_message}\n"
        f"  KL={metrics['approx_kl']:.4f} | clip_fraction={metrics['clip_fraction']:.3f}"
        f" | EV(before→after)={metrics['explained_variance_before']:.3f}"
        f"→{metrics['explained_variance_after']:.3f}"
        f" | std={metrics['std_mean']:.3f}"
        f" | 행동 [-1,1] 초과 비율={metrics['action_clip_fraction']:.1%}"
    )

### 지표 읽는 법

- **approx_kl, clip_fraction**: rollout 때 정책과 업데이트 후 정책의 변화
- **std_mean**: 연속 행동 정규분포의 평균 표준편차; 1 근처라면 각 차원의 행동이 꽤 넓게 샘플링될 수 있음
- **action_clip_fraction**: 게임에 전달하기 전에 잘라낸 행동 성분의 비율. 정책 클리핑 `clip_fraction`과 전혀 다름
- **explained_variance_before / after**: 이번 rollout의 Return 목표에 대해 Critic이 얼마나 맞추는지. 처음에 음수여도 이상하지 않으며 학습이 됐는지는 여러 rollout에서 봐야 함.

In [ ]:
if training_history:
    x = np.arange(1, len(training_history) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
    axes[0].plot(x, [m["explained_variance_before"] for m in training_history], marker="o", label="EV before")
    axes[0].plot(x, [m["explained_variance_after"] for m in training_history], marker="x", label="EV after")
    axes[0].axhline(0, linestyle=":")
    axes[0].set(title="Critic explained variance", xlabel="PPO iteration", ylabel="EV")
    axes[0].legend()
    axes[1].plot(x, [m["approx_kl"] for m in training_history], marker="o", label="approx KL")
    axes[1].plot(x, [m["clip_fraction"] for m in training_history], marker="x", label="policy clip fraction")
    axes[1].set(title="Policy update diagnostics", xlabel="PPO iteration", ylabel="value")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 9. 평가: 학습 없이 평균 행동으로 실행

확률적 탐색 `sample()` 대신, 분포의 평균 `distribution.mean`을 행동으로 쓴다. `deterministic=True`와 비슷한 평가 방식이다. 초기 시드를 고정해 반복 비교할 수 있다. 게임이 끝나거나 3,000 step에 도달하면 종료한다.

In [ ]:
def evaluate_policy(model: ActorCritic, seed: int = 123) -> dict:
    eval_env = make_environment()
    try:
        obs, _ = eval_env.reset(seed=seed)
        total_reward = 0.0
        while True:
            state = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                distribution, _ = model.distribution_and_value(state)
                action = distribution.mean.squeeze(0).cpu().numpy()
            action = np.clip(action, eval_env.action_space.low, eval_env.action_space.high).astype(np.float32)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            total_reward += float(reward)
            if terminated or truncated:
                return {
                    "seed": seed,
                    "total_reward": total_reward,
                    "score": info["score"],
                    "kills": info["kills"],
                    "environment_steps": info["steps"],
                    "game_frames": info["frames"],
                    "survival_seconds": info["frames"] / eval_env.game_config.fps,
                }
    finally:
        eval_env.close()

print(evaluate_policy(policy, seed=123))

## 10. 정리와 마무리

- **PPO는 policy만 있는 모델이 아니다.** `Actor + Critic + 경험 수집 + GAE + clip loss + Optimizer`를 관리하는 학습 절차다.
- **Gaussian**: Actor의 평균·표준편차를 이용해 현재 상태의 행동 분포를 만들고, 학습할 때 실수 행동을 샘플링한다.
- **explained_variance**: Critic의 예측이 실제 Return의 변화량을 얼마나 설명하는지 나타낸다.
- **두 가지 clip을 혼동하지 말 것**: `np.clip(action,-1,1)`은 게임 행동 한계이고, `clip_range=0.2`는 PPO 확률 비율 손실 제한이다.
- **주의할 실험 변수**: 현재 환경의 이동/발사/강화는 연속 수치에서 다시 이산 구간으로 변환되고, MLP에는 5,139차원 평탄화 상태가 들어간다.

원본 `train.py`의 모델 파일을 수정하거나 `PPO.load()`로 불러오지 않는다. 여기서 학습한 `policy`는 별도 교육용 네트워크다. 셀을 다시 처음부터 실행하려면 먼저 환경을 새로 생성해야 한다.

In [ ]:
# 모든 학습과 평가를 끝냈다면 환경 리소스를 정리한다.
environment.close()
print("학습 환경 정리 완료")